# AstroCLIMB — 5K Qwen3-VL QLoRA experiment

This notebook trains on exactly **5,000 balanced examples per epoch** (1,250 per class), selected across the complete label-sorted training CSV. A separate 400-row validation set follows the competition's approximate 10/30/30/30 class distribution. Because only 1,000 unique `same_figure` rows exist, that class is sampled with replacement after validation is reserved. Training uses **two-process DDP**, with one 4-bit Qwen3-VL-4B replica on each Kaggle T4.

Start a clean Kaggle session, select **GPU T4 x2**, attach the AstroCLIMB competition data, and enable Internet or attach the Qwen model. Training is launched as an external two-process job so each GPU worker starts in a clean interpreter.

In [1]:
# Keep Kaggle's torch, torchvision, Pillow, and scikit-learn unchanged.
%pip install -q --upgrade --upgrade-strategy only-if-needed "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes==0.47.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 584.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import base64
import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from sklearn.metrics import classification_report, confusion_matrix, f1_score

ImageFile.LOAD_TRUNCATED_IMAGES = True
csv.field_size_limit(sys.maxsize)

# Do not call torch.cuda.is_available(), device_count(), or any other CUDA function yet.
print('torch:', torch.__version__)
print('CUDA has not been initialized:', not torch.cuda.is_initialized())

torch: 2.10.0+cu128
CUDA has not been initialized: True


In [3]:
SEED = 42
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
TARGET_COLUMNS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
DIGIT_TO_LABEL = dict(enumerate(TARGET_COLUMNS))

TRAIN_QUOTAS = {0: 1250, 1: 1250, 2: 1250, 3: 1250}  # exactly 5,000 training examples
VAL_QUOTAS = {0: 40, 1: 120, 2: 120, 3: 120}     # natural 10/30/30/30 distribution
MAX_PIXELS = 448 * 448
MIN_PIXELS = 256 * 256
MAX_TEXT_CHARS = 3000
NUM_EPOCHS = 1
GRADIENT_ACCUMULATION = 8  # global batch = 1 x 2 GPUs x 8 = 16

WORK_ROOT = Path('/kaggle/working/astroclimb_5k') if Path('/kaggle/working').exists() else Path('./astroclimb_5k')
IMAGE_ROOT = WORK_ROOT / 'images'
TRAIN_MANIFEST = WORK_ROOT / 'train_5000.jsonl'
VAL_MANIFEST = WORK_ROOT / 'validation_400.jsonl'
ADAPTER_DIR = WORK_ROOT / 'adapter'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

def locate_csv(filename):
    preferred = Path('/kaggle/input/competitions/astroclimb') / filename
    if preferred.exists():
        return preferred
    preferred = Path('/kaggle/input/astroclimb') / filename
    if preferred.exists():
        return preferred
    roots = [Path('/kaggle/input'), Path('data')]
    candidates = [p for root in roots if root.exists() for p in root.rglob(filename)]
    candidates = sorted(candidates, key=lambda p: ('astroclimb' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f'{filename} not found; attach the AstroCLIMB data.')
    return candidates[0]

def locate_model():
    if Path('/kaggle/input').exists():
        configs = list(Path('/kaggle/input').rglob('config.json'))
        candidates = [p.parent for p in configs if 'qwen3' in str(p).lower() and 'vl' in str(p).lower()]
        if candidates:
            return str(sorted(candidates, key=lambda p: len(str(p)))[0])
    return MODEL_ID

TRAIN_CSV = locate_csv('train.csv')
TEST_CSV = locate_csv('test.csv')
MODEL_PATH = locate_model()
print('Train:', TRAIN_CSV)
print('Test:', TEST_CSV)
print('Model:', MODEL_PATH)
print('Working directory:', WORK_ROOT)

Train: /kaggle/input/competitions/astroclimb/train.csv
Test: /kaggle/input/competitions/astroclimb/test.csv
Model: Qwen/Qwen3-VL-4B-Instruct
Working directory: /kaggle/working/astroclimb_5k


## Select 5,000 train and 400 validation rows

Reservoir sampling scans all 10,000 rows, so examples are drawn throughout each label block rather than only from the start of a class. After validation is reserved, any class with fewer unique rows than its quota is completed by reproducible sampling with replacement.

In [4]:
def get_label(row):
    values = [int(float(row[c])) for c in TARGET_COLUMNS]
    if sum(values) != 1:
        raise ValueError(f'Invalid one-hot target for id={row.get("id")}: {values}')
    return values.index(1)

def reservoir_sample_by_class(path, capacities, seed=42):
    rng = random.Random(seed)
    reservoirs = {label: [] for label in range(4)}
    seen = {label: 0 for label in range(4)}
    started = time.perf_counter()
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        missing = {'id', 'obj_1', 'obj_2', *TARGET_COLUMNS} - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'Missing training columns: {sorted(missing)}')
        for row_index, row in enumerate(reader, start=1):
            label = get_label(row)
            seen[label] += 1
            compact = {'id': row['id'], 'obj_1': row['obj_1'], 'obj_2': row['obj_2'], 'label': label}
            bucket = reservoirs[label]
            capacity = capacities[label]
            if len(bucket) < capacity:
                bucket.append(compact)
            else:
                replacement = rng.randrange(seen[label])
                if replacement < capacity:
                    bucket[replacement] = compact
            if row_index % 1000 == 0:
                print(f'Scanned {row_index} rows in {(time.perf_counter() - started) / 60:.1f} min')
    print('Rows seen:', {DIGIT_TO_LABEL[k]: v for k, v in seen.items()})
    print(f'Complete CSV scan: {(time.perf_counter() - started) / 60:.2f} min')
    return reservoirs

capacities = {label: TRAIN_QUOTAS[label] + VAL_QUOTAS[label] for label in range(4)}
reservoirs = reservoir_sample_by_class(TRAIN_CSV, capacities, SEED)
rng = random.Random(SEED)
train_rows, val_rows = [], []
for label in range(4):
    rng.shuffle(reservoirs[label])
    val_n = VAL_QUOTAS[label]
    val_rows.extend(reservoirs[label][:val_n])
    available_train = reservoirs[label][val_n:]
    train_n = TRAIN_QUOTAS[label]
    if not available_train:
        raise RuntimeError(f'No training examples available for class {label}')
    if len(available_train) >= train_n:
        chosen_train = available_train[:train_n]
    else:
        chosen_train = available_train + rng.choices(available_train, k=train_n - len(available_train))
        print(f'{DIGIT_TO_LABEL[label]}: repeated {train_n - len(available_train)} examples to reach {train_n}')
    train_rows.extend(chosen_train)
rng.shuffle(train_rows)
rng.shuffle(val_rows)
assert len(train_rows) == 5000 and len(val_rows) == 400
print('Selected train rows:', len(train_rows), 'validation rows:', len(val_rows))

Scanned 1000 rows in 0.5 min
Scanned 2000 rows in 1.6 min
Scanned 3000 rows in 2.2 min
Scanned 4000 rows in 2.2 min
Scanned 5000 rows in 3.2 min
Scanned 6000 rows in 3.7 min
Scanned 7000 rows in 3.7 min
Scanned 8000 rows in 4.5 min
Scanned 9000 rows in 4.8 min
Scanned 10000 rows in 4.8 min
Rows seen: {'same_figure': 1000, 'same_paper': 3000, 'related_papers': 3000, 'unrelated_papers': 3000}
Complete CSV scan: 4.84 min
same_figure: repeated 290 examples to reach 1250
Selected train rows: 5000 validation rows: 400


## Decode and cache selected images

Images are resized once while preserving aspect ratio and saved as PNG. The manifests then contain small captions or local image paths instead of base64 blobs.

In [5]:
def looks_like_image(value):
    if not isinstance(value, str):
        return False
    value = value.lstrip()
    return value.startswith(('iVBORw0KGgo', '/9j/', 'UklGR', 'R0lGOD', 'data:image'))

def decode_image(value):
    value = value.strip()
    if value.startswith('data:image'):
        value = value.split(',', 1)[1]
    image = Image.open(io.BytesIO(base64.b64decode(value, validate=False)))
    image.load()
    return image.convert('RGB')

def resize_to_area(image, max_pixels=MAX_PIXELS):
    width, height = image.size
    if width * height <= max_pixels:
        return image
    scale = math.sqrt(max_pixels / (width * height))
    size = (max(1, round(width * scale)), max(1, round(height * scale)))
    return image.resize(size, Image.Resampling.LANCZOS)

def cache_object(value):
    if not looks_like_image(value):
        return {'kind': 'caption', 'value': value}
    digest = hashlib.sha256(value.encode('utf-8')).hexdigest()
    path = IMAGE_ROOT / f'{digest}.png'
    if not path.exists():
        image = resize_to_area(decode_image(value))
        image.save(path, format='PNG', compress_level=3)
    return {'kind': 'image', 'value': str(path)}

def write_manifest(rows, destination):
    started = time.perf_counter()
    modality_counts = {}
    with destination.open('w', encoding='utf-8') as handle:
        for index, row in enumerate(rows, start=1):
            obj_1 = cache_object(row['obj_1'])
            obj_2 = cache_object(row['obj_2'])
            mode = obj_1['kind'][0].upper() + obj_2['kind'][0].upper()
            modality_counts[mode] = modality_counts.get(mode, 0) + 1
            record = {'id': row['id'], 'obj_1': obj_1, 'obj_2': obj_2, 'label': int(row['label']), 'modality': mode}
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
            if index % 250 == 0:
                print(f'{destination.name}: cached {index}/{len(rows)}')
    print(destination.name, modality_counts, f'{(time.perf_counter() - started) / 60:.2f} min')

preprocess_started = time.perf_counter()
write_manifest(train_rows, TRAIN_MANIFEST)
write_manifest(val_rows, VAL_MANIFEST)
print(f'Total selected-data preprocessing: {(time.perf_counter() - preprocess_started) / 60:.2f} min')
print('Cached PNG files:', len(list(IMAGE_ROOT.glob('*.png'))))
del reservoirs, train_rows, val_rows
gc.collect()

train_5000.jsonl: cached 250/5000
train_5000.jsonl: cached 500/5000
train_5000.jsonl: cached 750/5000
train_5000.jsonl: cached 1000/5000
train_5000.jsonl: cached 1250/5000
train_5000.jsonl: cached 1500/5000
train_5000.jsonl: cached 1750/5000
train_5000.jsonl: cached 2000/5000
train_5000.jsonl: cached 2250/5000
train_5000.jsonl: cached 2500/5000
train_5000.jsonl: cached 2750/5000
train_5000.jsonl: cached 3000/5000
train_5000.jsonl: cached 3250/5000
train_5000.jsonl: cached 3500/5000
train_5000.jsonl: cached 3750/5000
train_5000.jsonl: cached 4000/5000
train_5000.jsonl: cached 4250/5000
train_5000.jsonl: cached 4500/5000
train_5000.jsonl: cached 4750/5000
train_5000.jsonl: cached 5000/5000
train_5000.jsonl {'II': 1287, 'CC': 1217, 'CI': 2496} 7.08 min
validation_400.jsonl: cached 250/400
validation_400.jsonl {'CC': 117, 'CI': 152, 'II': 131} 0.60 min
Total selected-data preprocessing: 7.67 min
Cached PNG files: 5014


0

## Shared prompt, dataset, and collator

In [6]:
SYSTEM_PROMPT = '''You classify the relationship between two objects from astronomy papers.
0: The objects are the figure and caption of the same scientific figure.
1: The objects are from different figures in the same paper.
2: The objects are from different papers and one paper cites the other.
3: The objects are from unrelated papers.
The relationship is symmetric. Output only one digit: 0, 1, 2, or 3.'''.strip()

def shorten_caption(text, max_chars=MAX_TEXT_CHARS):
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + '\n[...middle truncated...]\n' + text[-half:]

def object_content(number, obj):
    if obj['kind'] == 'image':
        with Image.open(obj['value']) as source:
            image = source.convert('RGB')
        return [
            {'type': 'text', 'text': f'Object {number} is a scientific figure:'},
            {'type': 'image', 'image': image},
        ]
    return [{'type': 'text', 'text': f'Object {number} is a figure caption:\n{shorten_caption(obj["value"])}'}]

def build_messages(row, include_answer, swap=False):
    obj_1, obj_2 = row['obj_1'], row['obj_2']
    if swap:
        obj_1, obj_2 = obj_2, obj_1
    content = object_content(1, obj_1) + object_content(2, obj_2)
    content.append({'type': 'text', 'text': 'Classify their relationship. Reply with one digit only.'})
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': content},
    ]
    if include_answer:
        messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': str(int(row['label']))}]})
    return messages

class ManifestDataset(torch.utils.data.Dataset):
    def __init__(self, path, random_swap=False):
        with Path(path).open('r', encoding='utf-8') as handle:
            self.rows = [json.loads(line) for line in handle]
        self.random_swap = random_swap
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = dict(self.rows[index])
        row['_swap'] = self.random_swap and random.random() < 0.5
        return row

class LabelOnlyCollator:
    def __init__(self, processor):
        self.processor = processor
        self.label_token_ids = []
        for digit in '0123':
            ids = processor.tokenizer.encode(digit, add_special_tokens=False)
            if len(ids) != 1:
                raise ValueError(f'Label {digit} is not a single token: {ids}')
            self.label_token_ids.append(ids[0])
    def __call__(self, features):
        if len(features) != 1:
            raise ValueError(f'Expected per-device batch 1, received {len(features)}')
        row = features[0]
        messages = build_messages(row, include_answer=True, swap=row.get('_swap', False))
        batch = self.processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors='pt'
        )
        target_id = self.label_token_ids[int(row['label'])]
        positions = torch.where(batch['input_ids'][0] == target_id)[0]
        if not len(positions):
            raise RuntimeError('Assistant label token not found.')
        labels = torch.full_like(batch['input_ids'], -100)
        labels[0, int(positions[-1])] = target_id
        batch['labels'] = labels
        return batch

## Two-T4 QLoRA training

Each external launcher process sees one T4 and receives a per-device batch of one. Using `accelerate launch` avoids inheriting the notebook kernel's CUDA state.

In [7]:
def train_worker(train_manifest, adapter_dir, model_id):
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration, Trainer, TrainingArguments
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    local_rank = int(os.environ.get('LOCAL_RANK', '0'))
    torch.cuda.set_device(local_rank)
    random.seed(SEED + local_rank)
    np.random.seed(SEED + local_rank)
    torch.manual_seed(SEED + local_rank)
    started = time.perf_counter()

    processor = AutoProcessor.from_pretrained(model_id, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    processor.tokenizer.padding_side = 'right'
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=quantization, dtype=torch.float16,
        attn_implementation='sdpa', device_map={'': local_rank},
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    ))
    if local_rank == 0:
        model.print_trainable_parameters()
        print(f'Model load on rank 0: {(time.perf_counter() - started) / 60:.2f} min')

    dataset = ManifestDataset(train_manifest, random_swap=True)
    args = TrainingArguments(
        output_dir=str(WORK_ROOT / 'trainer_output'),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=1e-4, warmup_ratio=0.05, lr_scheduler_type='cosine',
        weight_decay=0.01, max_grad_norm=1.0,
        fp16=True, bf16=False, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit', logging_steps=10,
        eval_strategy='no', save_strategy='no', report_to='none',
        remove_unused_columns=False, dataloader_num_workers=0,
        ddp_find_unused_parameters=False, seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=LabelOnlyCollator(processor))
    torch.cuda.synchronize()
    train_started = time.perf_counter()
    result = trainer.train()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - train_started

    if trainer.is_world_process_zero():
        adapter_path = Path(adapter_dir)
        adapter_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(adapter_path)
        processor.save_pretrained(adapter_path)
        metrics = dict(result.metrics)
        metrics.update({
            'wall_seconds': elapsed,
            'wall_minutes': elapsed / 60,
            'optimizer_steps': int(trainer.state.global_step),
            'seconds_per_optimizer_step': elapsed / max(1, trainer.state.global_step),
            'peak_gpu_gib_rank0': torch.cuda.max_memory_allocated() / 2**30,
        })
        with (adapter_path / 'training_metrics.json').open('w') as handle:
            json.dump(metrics, handle, indent=2)
        print(json.dumps(metrics, indent=2))

# Write a self-contained worker script, then launch fresh (spawn-safe) Python processes.
TRAIN_SCRIPT_PATH = WORK_ROOT / 'train_ddp.py'
TRAIN_SCRIPT_PATH.write_bytes(base64.b64decode('aW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UsIEltYWdlRmlsZQoKSW1hZ2VGaWxlLkxPQURfVFJVTkNBVEVEX0lNQUdFUyA9IFRydWUKClNFRUQgPSA0MgpNSU5fUElYRUxTID0gMjU2ICogMjU2Ck1BWF9QSVhFTFMgPSA0NDggKiA0NDgKTUFYX1RFWFRfQ0hBUlMgPSAzMDAwCgpTWVNURU1fUFJPTVBUID0gIiIiWW91IGNsYXNzaWZ5IHRoZSByZWxhdGlvbnNoaXAgYmV0d2VlbiB0d28gb2JqZWN0cyBmcm9tIGFzdHJvbm9teSBwYXBlcnMuCjA6IFRoZSBvYmplY3RzIGFyZSB0aGUgZmlndXJlIGFuZCBjYXB0aW9uIG9mIHRoZSBzYW1lIHNjaWVudGlmaWMgZmlndXJlLgoxOiBUaGUgb2JqZWN0cyBhcmUgZnJvbSBkaWZmZXJlbnQgZmlndXJlcyBpbiB0aGUgc2FtZSBwYXBlci4KMjogVGhlIG9iamVjdHMgYXJlIGZyb20gZGlmZmVyZW50IHBhcGVycyBhbmQgb25lIHBhcGVyIGNpdGVzIHRoZSBvdGhlci4KMzogVGhlIG9iamVjdHMgYXJlIGZyb20gdW5yZWxhdGVkIHBhcGVycy4KVGhlIHJlbGF0aW9uc2hpcCBpcyBzeW1tZXRyaWMuIE91dHB1dCBvbmx5IG9uZSBkaWdpdDogMCwgMSwgMiwgb3IgMy4iIiIuc3RyaXAoKQoKCmRlZiBzaG9ydGVuX2NhcHRpb24odGV4dCwgbWF4X2NoYXJzPU1BWF9URVhUX0NIQVJTKToKICAgIGlmIGxlbih0ZXh0KSA8PSBtYXhfY2hhcnM6CiAgICAgICAgcmV0dXJuIHRleHQKICAgIGhhbGYgPSBtYXhfY2hhcnMgLy8gMgogICAgcmV0dXJuIHRleHRbOmhhbGZdICsgIlxuWy4uLm1pZGRsZSB0cnVuY2F0ZWQuLi5dXG4iICsgdGV4dFstaGFsZjpdCgoKZGVmIG9iamVjdF9jb250ZW50KG51bWJlciwgb2JqKToKICAgIGlmIG9ialsia2luZCJdID09ICJpbWFnZSI6CiAgICAgICAgd2l0aCBJbWFnZS5vcGVuKG9ialsidmFsdWUiXSkgYXMgc291cmNlOgogICAgICAgICAgICBpbWFnZSA9IHNvdXJjZS5jb252ZXJ0KCJSR0IiKQogICAgICAgIHJldHVybiBbCiAgICAgICAgICAgIHsidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBmIk9iamVjdCB7bnVtYmVyfSBpcyBhIHNjaWVudGlmaWMgZmlndXJlOiJ9LAogICAgICAgICAgICB7InR5cGUiOiAiaW1hZ2UiLCAiaW1hZ2UiOiBpbWFnZX0sCiAgICAgICAgXQogICAgcmV0dXJuIFt7InR5cGUiOiAidGV4dCIsICJ0ZXh0IjogZiJPYmplY3Qge251bWJlcn0gaXMgYSBmaWd1cmUgY2FwdGlvbjpcbntzaG9ydGVuX2NhcHRpb24ob2JqWyd2YWx1ZSddKX0ifV0KCgpkZWYgYnVpbGRfbWVzc2FnZXMocm93LCBzd2FwPUZhbHNlKToKICAgIG9ial8xLCBvYmpfMiA9IHJvd1sib2JqXzEiXSwgcm93WyJvYmpfMiJdCiAgICBpZiBzd2FwOgogICAgICAgIG9ial8xLCBvYmpfMiA9IG9ial8yLCBvYmpfMQogICAgY29udGVudCA9IG9iamVjdF9jb250ZW50KDEsIG9ial8xKSArIG9iamVjdF9jb250ZW50KDIsIG9ial8yKQogICAgY29udGVudC5hcHBlbmQoeyJ0eXBlIjogInRleHQiLCAidGV4dCI6ICJDbGFzc2lmeSB0aGVpciByZWxhdGlvbnNoaXAuIFJlcGx5IHdpdGggb25lIGRpZ2l0IG9ubHkuIn0pCiAgICByZXR1cm4gWwogICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IFt7InR5cGUiOiAidGV4dCIsICJ0ZXh0IjogU1lTVEVNX1BST01QVH1dfSwKICAgICAgICB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogY29udGVudH0sCiAgICAgICAgeyJyb2xlIjogImFzc2lzdGFudCIsICJjb250ZW50IjogW3sidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBzdHIoaW50KHJvd1sibGFiZWwiXSkpfV19LAogICAgXQoKCmNsYXNzIE1hbmlmZXN0RGF0YXNldCh0b3JjaC51dGlscy5kYXRhLkRhdGFzZXQpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhdGgsIG92ZXJzYW1wbGVfc2FtZV9maWd1cmU9RmFsc2UpOgogICAgICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgICAgICBzZWxmLnJvd3MgPSBbanNvbi5sb2FkcyhsaW5lKSBmb3IgbGluZSBpbiBoYW5kbGVdCiAgICAgICAgaWYgb3ZlcnNhbXBsZV9zYW1lX2ZpZ3VyZToKICAgICAgICAgICAgY2xhc3NfemVybyA9IFtyb3cgZm9yIHJvdyBpbiBzZWxmLnJvd3MgaWYgcm93WyJsYWJlbCJdID09IDBdCiAgICAgICAgICAgIHNlbGYucm93cy5leHRlbmQoY2xhc3NfemVybykKICAgICAgICAgICAgc2VsZi5yb3dzLmV4dGVuZChjbGFzc196ZXJvKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi5yb3dzKQoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleCk6CiAgICAgICAgcm93ID0gZGljdChzZWxmLnJvd3NbaW5kZXhdKQogICAgICAgIHJvd1siX3N3YXAiXSA9IHJhbmRvbS5yYW5kb20oKSA8IDAuNQogICAgICAgIHJldHVybiByb3cKCgpjbGFzcyBMYWJlbE9ubHlDb2xsYXRvcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwcm9jZXNzb3IpOgogICAgICAgIHNlbGYucHJvY2Vzc29yID0gcHJvY2Vzc29yCiAgICAgICAgc2VsZi5sYWJlbF90b2tlbl9pZHMgPSBbXQogICAgICAgIGZvciBkaWdpdCBpbiAiMDEyMyI6CiAgICAgICAgICAgIGlkcyA9IHByb2Nlc3Nvci50b2tlbml6ZXIuZW5jb2RlKGRpZ2l0LCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpCiAgICAgICAgICAgIGlmIGxlbihpZHMpICE9IDE6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTGFiZWwge2RpZ2l0fSBpcyBub3QgYSBzaW5nbGUgdG9rZW46IHtpZHN9IikKICAgICAgICAgICAgc2VsZi5sYWJlbF90b2tlbl9pZHMuYXBwZW5kKGlkc1swXSkKCiAgICBkZWYgX19jYWxsX18oc2VsZiwgZmVhdHVyZXMpOgogICAgICAgIGlmIGxlbihmZWF0dXJlcykgIT0gMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIHBlci1kZXZpY2UgYmF0Y2ggMSwgcmVjZWl2ZWQge2xlbihmZWF0dXJlcyl9IikKICAgICAgICByb3cgPSBmZWF0dXJlc1swXQogICAgICAgIG1lc3NhZ2VzID0gYnVpbGRfbWVzc2FnZXMocm93LCBzd2FwPXJvdy5nZXQoIl9zd2FwIiwgRmFsc2UpKQogICAgICAgIGJhdGNoID0gc2VsZi5wcm9jZXNzb3IuYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICAgICAgbWVzc2FnZXMsCiAgICAgICAgICAgIHRva2VuaXplPVRydWUsCiAgICAgICAgICAgIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1GYWxzZSwKICAgICAgICAgICAgcmV0dXJuX2RpY3Q9VHJ1ZSwKICAgICAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IiwKICAgICAgICApCiAgICAgICAgdGFyZ2V0X2lkID0gc2VsZi5sYWJlbF90b2tlbl9pZHNbaW50KHJvd1sibGFiZWwiXSldCiAgICAgICAgcG9zaXRpb25zID0gdG9yY2gud2hlcmUoYmF0Y2hbImlucHV0X2lkcyJdWzBdID09IHRhcmdldF9pZClbMF0KICAgICAgICBpZiBub3QgbGVuKHBvc2l0aW9ucyk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQXNzaXN0YW50IGxhYmVsIHRva2VuIG5vdCBmb3VuZC4iKQogICAgICAgIGxhYmVscyA9IHRvcmNoLmZ1bGxfbGlrZShiYXRjaFsiaW5wdXRfaWRzIl0sIC0xMDApCiAgICAgICAgbGFiZWxzWzAsIGludChwb3NpdGlvbnNbLTFdKV0gPSB0YXJnZXRfaWQKICAgICAgICBiYXRjaFsibGFiZWxzIl0gPSBsYWJlbHMKICAgICAgICByZXR1cm4gYmF0Y2gKCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10cmFpbi1tYW5pZmVzdCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWFkYXB0ZXItZGlyIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0td29yay1yb290IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwtcGF0aCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MS4wKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ncmFkaWVudC1hY2N1bXVsYXRpb24iLCB0eXBlPWludCwgZGVmYXVsdD04KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdmVyc2FtcGxlLXNhbWUtZmlndXJlIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2F2ZS1zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKICAgICMgU2VsZWN0IHRoaXMgcmFuaydzIEdQVSBiZWZvcmUgVHJhbnNmb3JtZXJzL3RvcmNoYW8gY2FuIHByb2JlIGFuZCBpbml0aWFsaXplIENVREEuCiAgICBsb2NhbF9yYW5rID0gaW50KG9zLmVudmlyb24uZ2V0KCJMT0NBTF9SQU5LIiwgIjAiKSkKICAgIHRvcmNoLmN1ZGEuc2V0X2RldmljZShsb2NhbF9yYW5rKQoKICAgICMgVGhlc2UgaW1wb3J0cyBoYXBwZW4gb25seSBhZnRlciBhY2NlbGVyYXRlIGhhcyBjcmVhdGVkIGNsZWFuIHdvcmtlciBwcm9jZXNzZXMuCiAgICBmcm9tIHBlZnQgaW1wb3J0IExvcmFDb25maWcsIGdldF9wZWZ0X21vZGVsLCBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgKAogICAgICAgIEF1dG9Qcm9jZXNzb3IsCiAgICAgICAgQml0c0FuZEJ5dGVzQ29uZmlnLAogICAgICAgIFF3ZW4zVkxGb3JDb25kaXRpb25hbEdlbmVyYXRpb24sCiAgICAgICAgVHJhaW5lciwKICAgICAgICBUcmFpbmluZ0FyZ3VtZW50cywKICAgICkKCiAgICByYW5kb20uc2VlZChTRUVEICsgbG9jYWxfcmFuaykKICAgIG5wLnJhbmRvbS5zZWVkKFNFRUQgKyBsb2NhbF9yYW5rKQogICAgdG9yY2gubWFudWFsX3NlZWQoU0VFRCArIGxvY2FsX3JhbmspCiAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQoKICAgIHByb2Nlc3NvciA9IEF1dG9Qcm9jZXNzb3IuZnJvbV9wcmV0cmFpbmVkKGFyZ3MubW9kZWxfcGF0aCwgbWluX3BpeGVscz1NSU5fUElYRUxTLCBtYXhfcGl4ZWxzPU1BWF9QSVhFTFMpCiAgICBwcm9jZXNzb3IudG9rZW5pemVyLnBhZGRpbmdfc2lkZSA9ICJyaWdodCIKICAgIHF1YW50aXphdGlvbiA9IEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPSJuZjQiLAogICAgICAgIGJuYl80Yml0X3VzZV9kb3VibGVfcXVhbnQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPXRvcmNoLmZsb2F0MTYsCiAgICApCiAgICBtb2RlbCA9IFF3ZW4zVkxGb3JDb25kaXRpb25hbEdlbmVyYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIGFyZ3MubW9kZWxfcGF0aCwKICAgICAgICBxdWFudGl6YXRpb25fY29uZmlnPXF1YW50aXphdGlvbiwKICAgICAgICBkdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgIGF0dG5faW1wbGVtZW50YXRpb249InNkcGEiLAogICAgICAgIGRldmljZV9tYXA9eyIiOiBsb2NhbF9yYW5rfSwKICAgICkKICAgIG1vZGVsLmNvbmZpZy51c2VfY2FjaGUgPSBGYWxzZQogICAgbW9kZWwgPSBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nKG1vZGVsLCB1c2VfZ3JhZGllbnRfY2hlY2twb2ludGluZz1UcnVlKQogICAgbW9kZWwgPSBnZXRfcGVmdF9tb2RlbCgKICAgICAgICBtb2RlbCwKICAgICAgICBMb3JhQ29uZmlnKAogICAgICAgICAgICByPTE2LAogICAgICAgICAgICBsb3JhX2FscGhhPTMyLAogICAgICAgICAgICBsb3JhX2Ryb3BvdXQ9MC4wNSwKICAgICAgICAgICAgYmlhcz0ibm9uZSIsCiAgICAgICAgICAgIHRhc2tfdHlwZT0iQ0FVU0FMX0xNIiwKICAgICAgICAgICAgdGFyZ2V0X21vZHVsZXM9WyJxX3Byb2oiLCAia19wcm9qIiwgInZfcHJvaiIsICJvX3Byb2oiXSwKICAgICAgICApLAogICAgKQogICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgIG1vZGVsLnByaW50X3RyYWluYWJsZV9wYXJhbWV0ZXJzKCkKICAgICAgICBwcmludChmIk1vZGVsIGxvYWQgb24gcmFuayAwOiB7KHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkKSAvIDYwOi4yZn0gbWluIiwgZmx1c2g9VHJ1ZSkKCiAgICBkYXRhc2V0ID0gTWFuaWZlc3REYXRhc2V0KGFyZ3MudHJhaW5fbWFuaWZlc3QsIG92ZXJzYW1wbGVfc2FtZV9maWd1cmU9YXJncy5vdmVyc2FtcGxlX3NhbWVfZmlndXJlKQogICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgIHByaW50KGYiRWZmZWN0aXZlIHRyYWluaW5nIHJvd3M6IHtsZW4oZGF0YXNldCl9IiwgZmx1c2g9VHJ1ZSkKICAgIHRyYWluaW5nX2FyZ3MgPSBUcmFpbmluZ0FyZ3VtZW50cygKICAgICAgICBvdXRwdXRfZGlyPXN0cihQYXRoKGFyZ3Mud29ya19yb290KSAvICJ0cmFpbmVyX291dHB1dCIpLAogICAgICAgIHBlcl9kZXZpY2VfdHJhaW5fYmF0Y2hfc2l6ZT0xLAogICAgICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcz1hcmdzLmdyYWRpZW50X2FjY3VtdWxhdGlvbiwKICAgICAgICBudW1fdHJhaW5fZXBvY2hzPWFyZ3MuZXBvY2hzLAogICAgICAgIGxlYXJuaW5nX3JhdGU9MWUtNCwKICAgICAgICB3YXJtdXBfcmF0aW89MC4wNSwKICAgICAgICBscl9zY2hlZHVsZXJfdHlwZT0iY29zaW5lIiwKICAgICAgICB3ZWlnaHRfZGVjYXk9MC4wMSwKICAgICAgICBtYXhfZ3JhZF9ub3JtPTEuMCwKICAgICAgICBmcDE2PVRydWUsCiAgICAgICAgYmYxNj1GYWxzZSwKICAgICAgICBncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUsCiAgICAgICAgZ3JhZGllbnRfY2hlY2twb2ludGluZ19rd2FyZ3M9eyJ1c2VfcmVlbnRyYW50IjogRmFsc2V9LAogICAgICAgIG9wdGltPSJwYWdlZF9hZGFtd184Yml0IiwKICAgICAgICBsb2dnaW5nX3N0ZXBzPTEwLAogICAgICAgIGV2YWxfc3RyYXRlZ3k9Im5vIiwKICAgICAgICBzYXZlX3N0cmF0ZWd5PSJzdGVwcyIgaWYgYXJncy5zYXZlX3N0ZXBzID4gMCBlbHNlICJubyIsCiAgICAgICAgc2F2ZV9zdGVwcz1tYXgoMSwgYXJncy5zYXZlX3N0ZXBzKSwKICAgICAgICBzYXZlX3RvdGFsX2xpbWl0PTIsCiAgICAgICAgcmVwb3J0X3RvPSJub25lIiwKICAgICAgICByZW1vdmVfdW51c2VkX2NvbHVtbnM9RmFsc2UsCiAgICAgICAgZGF0YWxvYWRlcl9udW1fd29ya2Vycz0wLAogICAgICAgIGRkcF9maW5kX3VudXNlZF9wYXJhbWV0ZXJzPUZhbHNlLAogICAgICAgIHNlZWQ9U0VFRCwKICAgICkKICAgIHRyYWluZXIgPSBUcmFpbmVyKAogICAgICAgIG1vZGVsPW1vZGVsLAogICAgICAgIGFyZ3M9dHJhaW5pbmdfYXJncywKICAgICAgICB0cmFpbl9kYXRhc2V0PWRhdGFzZXQsCiAgICAgICAgZGF0YV9jb2xsYXRvcj1MYWJlbE9ubHlDb2xsYXRvcihwcm9jZXNzb3IpLAogICAgKQogICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICB0cmFpbl9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgcmVzdWx0ID0gdHJhaW5lci50cmFpbigpCiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gdHJhaW5fc3RhcnRlZAoKICAgIGlmIHRyYWluZXIuaXNfd29ybGRfcHJvY2Vzc196ZXJvKCk6CiAgICAgICAgYWRhcHRlcl9wYXRoID0gUGF0aChhcmdzLmFkYXB0ZXJfZGlyKQogICAgICAgIGFkYXB0ZXJfcGF0aC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgbW9kZWwuc2F2ZV9wcmV0cmFpbmVkKGFkYXB0ZXJfcGF0aCkKICAgICAgICBwcm9jZXNzb3Iuc2F2ZV9wcmV0cmFpbmVkKGFkYXB0ZXJfcGF0aCkKICAgICAgICBtZXRyaWNzID0gZGljdChyZXN1bHQubWV0cmljcykKICAgICAgICBtZXRyaWNzLnVwZGF0ZSgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgIndhbGxfc2Vjb25kcyI6IGVsYXBzZWQsCiAgICAgICAgICAgICAgICAid2FsbF9taW51dGVzIjogZWxhcHNlZCAvIDYwLAogICAgICAgICAgICAgICAgIm9wdGltaXplcl9zdGVwcyI6IGludCh0cmFpbmVyLnN0YXRlLmdsb2JhbF9zdGVwKSwKICAgICAgICAgICAgICAgICJzZWNvbmRzX3Blcl9vcHRpbWl6ZXJfc3RlcCI6IGVsYXBzZWQgLyBtYXgoMSwgdHJhaW5lci5zdGF0ZS5nbG9iYWxfc3RlcCksCiAgICAgICAgICAgICAgICAicGVha19ncHVfZ2liX3JhbmswIjogdG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpIC8gMioqMzAsCiAgICAgICAgICAgICAgICAiZWZmZWN0aXZlX3RyYWluaW5nX3Jvd3MiOiBsZW4oZGF0YXNldCksCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICAgICAgd2l0aCAoYWRhcHRlcl9wYXRoIC8gInRyYWluaW5nX21ldHJpY3MuanNvbiIpLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIGpzb24uZHVtcChtZXRyaWNzLCBoYW5kbGUsIGluZGVudD0yKQogICAgICAgIHByaW50KGpzb24uZHVtcHMobWV0cmljcywgaW5kZW50PTIpLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'))
command = [
    sys.executable, '-m', 'accelerate.commands.launch',
    '--multi_gpu', '--num_processes', '2',
    str(TRAIN_SCRIPT_PATH),
    '--train-manifest', str(TRAIN_MANIFEST),
    '--adapter-dir', str(ADAPTER_DIR),
    '--work-root', str(WORK_ROOT),
    '--model-path', MODEL_PATH,
    '--epochs', str(NUM_EPOCHS),
    '--gradient-accumulation', str(GRADIENT_ACCUMULATION),
]
print('Launching:', ' '.join(command), flush=True)
launch_started = time.perf_counter()
launch_env = dict(os.environ, PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false')
subprocess.run(command, check=True, env=launch_env)
print(f'Total two-process launch wall time: {(time.perf_counter() - launch_started) / 60:.2f} min')

Launching: /usr/bin/python3 -m accelerate.commands.launch --multi_gpu --num_processes 2 /kaggle/working/astroclimb_5k/train_ddp.py --train-manifest /kaggle/working/astroclimb_5k/train_5000.jsonl --adapter-dir /kaggle/working/astroclimb_5k/adapter --work-root /kaggle/working/astroclimb_5k --model-path Qwen/Qwen3-VL-4B-Instruct --epochs 1 --gradient-accumulation 8


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.42s/it]


trainable params: 11,796,480 || all params: 4,449,612,288 || trainable%: 0.2651
Model load on rank 0: 1.15 min
Effective training rows: 5000


  3%|▎         | 10/313 [02:08<1:07:44, 13.41s/it]

{'loss': 4.14, 'grad_norm': 24.068321228027344, 'learning_rate': 2.5e-05, 'epoch': 0.03}


...

{'loss': 0.6378, 'grad_norm': 5.623135566711426, 'learning_rate': 2.2640387134577058e-07, 'epoch': 0.99}


100%|██████████| 313/313 [1:11:41<00:00, 13.74s/it]


{'train_runtime': 4302.1371, 'train_samples_per_second': 1.162, 'train_steps_per_second': 0.073, 'train_loss': 0.8769082345139866, 'epoch': 1.0}
{
  "train_runtime": 4302.1371,
  "train_samples_per_second": 1.162,
  "train_steps_per_second": 0.073,
  "total_flos": 5.413275297854259e+16,
  "train_loss": 0.8769082345139866,
  "epoch": 1.0,
  "wall_seconds": 4303.135630211,
  "wall_minutes": 71.71892717018333,
  "optimizer_steps": 313,
  "seconds_per_optimizer_step": 13.748037157223642,
  "peak_gpu_gib_rank0": 6.342045783996582,
  "effective_training_rows": 5000
}


[rank0]:[W915 06:22:09.596775256 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Total two-process launch wall time: 73.46 min


## Load the adapter on one T4 and validate

In [8]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import PeftModel

print('Visible GPUs after launcher:', torch.cuda.device_count())
processor = AutoProcessor.from_pretrained(ADAPTER_DIR, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_PATH, quantization_config=quantization, dtype=torch.float16,
    attn_implementation='sdpa', device_map={'': 0},
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
model.config.use_cache = True
label_token_ids = [processor.tokenizer.encode(x, add_special_tokens=False)[0] for x in '0123']

@torch.inference_mode()
def predict_probabilities(row, swap=False, apply_modality_mask=False):
    messages = build_messages(row, include_answer=False, swap=swap)
    batch = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt'
    )
    batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
    logits = model(**batch).logits[0, -1, label_token_ids].float()
    if apply_modality_mask and row.get('modality') in {'CC', 'II'}:
        logits[0] = float('-inf')
    return torch.softmax(logits, dim=-1).cpu().numpy()

validation = ManifestDataset(VAL_MANIFEST, random_swap=False).rows
torch.cuda.synchronize()
validation_started = time.perf_counter()
val_probabilities = np.stack([predict_probabilities(row) for row in validation])
torch.cuda.synchronize()
validation_seconds = time.perf_counter() - validation_started
y_true = np.array([row['label'] for row in validation])
y_pred = val_probabilities.argmax(axis=1)
print(f'Validation inference: {validation_seconds / 60:.2f} min, {validation_seconds / len(validation):.3f}s/row')
print(f'Projected one-T4 inference for 10,000 rows: {validation_seconds / len(validation) * 10000 / 3600:.2f}h')
print('Macro-F1:', f1_score(y_true, y_pred, average='macro'))
print(classification_report(y_true, y_pred, labels=range(4), target_names=TARGET_COLUMNS, digits=4, zero_division=0))
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=range(4)),
    index=[f'true_{x}' for x in TARGET_COLUMNS],
    columns=[f'pred_{x}' for x in TARGET_COLUMNS],
))
np.save(WORK_ROOT / 'validation_probabilities.npy', val_probabilities)

Visible GPUs after launcher: 2


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Validation inference: 3.87 min, 0.581s/row
Projected one-T4 inference for 10,000 rows: 1.61h
Macro-F1: 0.6936544369272779
                  precision    recall  f1-score   support

     same_figure     0.8667    0.9750    0.9176        40
      same_paper     0.6800    0.5667    0.6182       120
  related_papers     0.4786    0.5583    0.5154       120
unrelated_papers     0.7391    0.7083    0.7234       120

        accuracy                         0.6475       400
       macro avg     0.6911    0.7021    0.6937       400
    weighted avg     0.6560    0.6475    0.6489       400



,pred_same_figure,pred_same_paper,pred_related_papers,pred_unrelated_papers
true_same_figure,39,0,1,0
true_same_paper,4,68,37,11
true_related_papers,2,32,67,19
true_unrelated_papers,0,0,35,85


## Optional test timing and submission

The default processes 100 rows to measure inference time. Change `TEST_LIMIT=None` only when you deliberately want a complete one-T4 submission; that may take several hours.

In [9]:
TEST_LIMIT = None  # Set to None for all 10,000 rows.
TEST_LOG_EVERY = 25
USE_SWAP_TTA = False
APPLY_MODALITY_MASK = False  # Keep false for the pure language-LoRA baseline.
SUBMISSION_PATH = WORK_ROOT / 'submission.csv'
PROBABILITY_PATH = WORK_ROOT / 'submission_probabilities.csv'

def raw_test_object(value):
    if looks_like_image(value):
        return {'kind': 'image_pil', 'value': resize_to_area(decode_image(value))}
    return {'kind': 'caption', 'value': value}

def test_object_content(number, obj):
    if obj['kind'] == 'image_pil':
        return [
            {'type': 'text', 'text': f'Object {number} is a scientific figure:'},
            {'type': 'image', 'image': obj['value']},
        ]
    return [{'type': 'text', 'text': f'Object {number} is a figure caption:\n{shorten_caption(obj["value"])}'}]

def convert_test_row(raw):
    obj_1, obj_2 = raw_test_object(raw['obj_1']), raw_test_object(raw['obj_2'])
    mode = ('I' if obj_1['kind'].startswith('image') else 'C') + ('I' if obj_2['kind'].startswith('image') else 'C')
    return {'id': raw['id'], 'obj_1': obj_1, 'obj_2': obj_2, 'modality': mode}

# Temporarily make object_content accept in-memory test images.
manifest_object_content = object_content
def object_content(number, obj):
    if obj['kind'] == 'image_pil':
        return test_object_content(number, obj)
    return manifest_object_content(number, obj)

rows_written = 0
counts = np.zeros(4, dtype=int)
test_started = time.perf_counter()
with (
    TEST_CSV.open('r', encoding='utf-8', newline='') as source,
    SUBMISSION_PATH.open('w', encoding='utf-8', newline='') as output,
    PROBABILITY_PATH.open('w', encoding='utf-8', newline='') as prob_output,
):
    reader = csv.DictReader(source)
    writer = csv.DictWriter(output, fieldnames=['id', *TARGET_COLUMNS])
    prob_writer = csv.DictWriter(prob_output, fieldnames=['id', *[f'p_{x}' for x in TARGET_COLUMNS]])
    writer.writeheader(); prob_writer.writeheader()
    for index, raw in enumerate(reader):
        if TEST_LIMIT is not None and index >= TEST_LIMIT:
            break
        row = convert_test_row(raw)
        p = predict_probabilities(row, apply_modality_mask=APPLY_MODALITY_MASK)
        if USE_SWAP_TTA:
            p = 0.5 * (p + predict_probabilities(row, swap=True, apply_modality_mask=APPLY_MODALITY_MASK))
        prediction = int(p.argmax())
        writer.writerow({'id': row['id'], **{name: int(i == prediction) for i, name in enumerate(TARGET_COLUMNS)}})
        prob_writer.writerow({'id': row['id'], **{f'p_{name}': float(p[i]) for i, name in enumerate(TARGET_COLUMNS)}})
        counts[prediction] += 1
        rows_written += 1
        if rows_written % TEST_LOG_EVERY == 0:
            elapsed = time.perf_counter() - test_started
            seconds_per_row = elapsed / rows_written
            total = TEST_LIMIT if TEST_LIMIT is not None else 10000
            print(f'{rows_written}/{total} | {seconds_per_row:.3f}s/row | projected 10K={seconds_per_row * 10000 / 3600:.2f}h')

elapsed = time.perf_counter() - test_started
print(f'Wrote {rows_written} rows in {elapsed / 60:.2f} min; counts={counts.tolist()}')
submission = pd.read_csv(SUBMISSION_PATH)
assert submission.columns.tolist() == ['id', *TARGET_COLUMNS]
assert submission['id'].is_unique
assert submission[TARGET_COLUMNS].isin([0, 1]).all().all()
assert (submission[TARGET_COLUMNS].sum(axis=1) == 1).all()
if TEST_LIMIT is None:
    assert len(submission) == 10000
    print('Complete submission ready:', SUBMISSION_PATH)
else:
    print('Timing-only partial submission. Set TEST_LIMIT=None for the complete file.')
display(submission.head())

25/10000 | 0.643s/row | projected 10K=1.79h
50/10000 | 0.668s/row | projected 10K=1.86h
75/10000 | 0.679s/row | projected 10K=1.89h
100/10000 | 0.668s/row | projected 10K=1.86h
125/10000 | 0.667s/row | projected 10K=1.85h
150/10000 | 0.664s/row | projected 10K=1.84h
175/10000 | 0.657s/row | projected 10K=1.83h
200/10000 | 0.655s/row | projected 10K=1.82h
225/10000 | 0.657s/row | projected 10K=1.83h
250/10000 | 0.658s/row | projected 10K=1.83h
275/10000 | 0.657s/row | projected 10K=1.82h
300/10000 | 0.655s/row | projected 10K=1.82h
325/10000 | 0.657s/row | projected 10K=1.82h
350/10000 | 0.657s/row | projected 10K=1.83h
375/10000 | 0.655s/row | projected 10K=1.82h
400/10000 | 0.658s/row | projected 10K=1.83h
425/10000 | 0.658s/row | projected 10K=1.83h
450/10000 | 0.659s/row | projected 10K=1.83h
475/10000 | 0.660s/row | projected 10K=1.83h
500/10000 | 0.659s/row | projected 10K=1.83h
525/10000 | 0.660s/row | projected 10K=1.83h
550/10000 | 0.660s/row | projected 10K=1.83h
575/10000 | 0

,id,same_figure,same_paper,related_papers,unrelated_papers
0,0,1,0,0,0
1,1,1,0,0,0
2,2,1,0,0,0
3,3,1,0,0,0
4,4,1,0,0,0
